# Solucion BI para LogiAndes S.A.

Este notebook desarrolla el procesamiento, analisis y documentacion de KPIs para una solucion de Business Intelligence orientada a la operacion logistica de LogiAndes S.A. Se complementa con el prototipo en Streamlit disponible en `presentacion_logiandes.py`.

## 1. Problema empresarial y audiencias

LogiAndes necesita monitorear la calidad de su servicio logistico, identificar demoras, reclamos, territorios con friccion y valor comercial expuesto. La solucion BI debe apoyar decisiones para:

- **Gerencia general:** seguimiento del desempeno global y cumplimiento de metas.
- **Operaciones:** deteccion de cuellos de botella por canal, provincia, distancia y pedidos criticos.
- **Comercial:** revision del impacto de la promesa de entrega sobre ventas y experiencia.
- **Finanzas:** priorizacion de ventas asociadas a pedidos con riesgo logistico.
- **Atencion al cliente:** seguimiento de reclamos, motivos frecuentes y pedidos que requieren gestion.

In [ ]:
from pathlib import Path

import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

BASE_DIR = Path.cwd()
DATA_PATH = BASE_DIR / 'logiandes.csv'

df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
df.info()
df.describe(include='all').T

## 2. Preparacion de datos

Se convierten fechas, se normalizan textos con problemas de codificacion y se crean variables operativas para el dashboard: ciclo total, horas de entrega, venta por kilometro y marca de reclamo.

In [ ]:
def preparar_datos(datos: pd.DataFrame) -> pd.DataFrame:
    datos = datos.copy()
    datos['fecha_pedido'] = pd.to_datetime(datos['fecha_pedido'], errors='coerce')
    reemplazos = {
        'Est�ndar urbano': 'Estandar urbano',
        'Estándar urbano': 'Estandar urbano',
        'Manab�': 'Manabi',
        'Manabí': 'Manabi',
    }
    for columna in ['provincia', 'region', 'canal_entrega', 'tipo_cliente', 'motivo_reclamo']:
        datos[columna] = datos[columna].replace(reemplazos)
    datos['motivo_reclamo'] = datos['motivo_reclamo'].fillna('Sin reclamo')
    datos['fecha'] = datos['fecha_pedido'].dt.date
    datos['hora'] = datos['fecha_pedido'].dt.hour
    datos['mes'] = datos['fecha_pedido'].dt.to_period('M').astype(str)
    datos['ciclo_total_min'] = datos['tiempo_atencion_min'] + datos['tiempo_individual_entrega_min']
    datos['entrega_horas'] = datos['tiempo_individual_entrega_min'] / 60
    datos['venta_por_km'] = datos['ventas_asociadas_usd'] / datos['distancia_km'].replace(0, pd.NA)
    datos['tuvo_reclamo'] = datos['reclamos_registrados'].gt(0)
    return datos

datos = preparar_datos(df)
datos.head()

## 3. KPIs empresariales

Se definen y calculan indicadores que conectan la operacion logistica con decisiones de servicio, territorio, reclamos y valor comercial.

In [ ]:
SLA_MIN = 150

def calcular_kpis(datos: pd.DataFrame, sla_min: int = SLA_MIN) -> pd.DataFrame:
    pedidos = len(datos)
    reclamos = int(datos['reclamos_registrados'].sum())
    fuera_sla = datos['tiempo_individual_entrega_min'].gt(sla_min)
    ventas_riesgo = datos.loc[fuera_sla | datos['tuvo_reclamo'], 'ventas_asociadas_usd'].sum()
    filas = [
        {
            'KPI': 'Cumplimiento de SLA',
            'Valor': 1 - fuera_sla.mean(),
            'Formato': 'porcentaje',
            'Meta': '>= 85%',
            'Estado': 'En meta' if 1 - fuera_sla.mean() >= 0.85 else 'Revisar',
        },
        {
            'KPI': 'P90 de tiempo de entrega',
            'Valor': datos['tiempo_individual_entrega_min'].quantile(0.90),
            'Formato': 'minutos',
            'Meta': '<= 190 min',
            'Estado': 'En meta' if datos['tiempo_individual_entrega_min'].quantile(0.90) <= 190 else 'Revisar',
        },
        {
            'KPI': 'Tasa de reclamos',
            'Valor': reclamos / pedidos if pedidos else 0,
            'Formato': 'porcentaje',
            'Meta': '<= 3%',
            'Estado': 'En meta' if (reclamos / pedidos if pedidos else 0) <= 0.03 else 'Revisar',
        },
        {
            'KPI': 'Distancia promedio por pedido',
            'Valor': datos['distancia_km'].mean(),
            'Formato': 'km',
            'Meta': 'Monitoreo por territorio',
            'Estado': 'Controlar por territorio',
        },
        {
            'KPI': 'Ventas en riesgo logistico',
            'Valor': ventas_riesgo,
            'Formato': 'usd',
            'Meta': 'Reducir tendencia',
            'Estado': 'Priorizar' if ventas_riesgo > 0 else 'Sin riesgo',
        },
    ]
    return pd.DataFrame(filas)

kpis = calcular_kpis(datos)
kpis

In [ ]:
ficha_kpis = pd.DataFrame([
    {
        'Nombre': 'Cumplimiento de SLA',
        'Objetivo': 'Medir la proporcion de pedidos entregados dentro del tiempo objetivo.',
        'Formula': f'Pedidos con entrega <= {SLA_MIN} min / total de pedidos',
        'Fuente': 'logiandes.csv: tiempo_individual_entrega_min, pedido_id',
        'Frecuencia': 'Diaria / semanal',
        'Meta': '>= 85%',
        'Umbral': '< 75% requiere revision operativa',
        'Interpretacion': 'Valores bajos senalan saturacion, promesa comercial agresiva o problemas de ruta.',
    },
    {
        'Nombre': 'P90 de tiempo de entrega',
        'Objetivo': 'Controlar la experiencia de los pedidos mas lentos.',
        'Formula': 'Percentil 90 de tiempo_individual_entrega_min',
        'Fuente': 'logiandes.csv: tiempo_individual_entrega_min',
        'Frecuencia': 'Diaria / semanal',
        'Meta': '<= 190 min',
        'Umbral': '> 210 min implica foco critico',
        'Interpretacion': 'Resume la cola de demoras mejor que el promedio cuando hay casos extremos.',
    },
    {
        'Nombre': 'Tasa de reclamos',
        'Objetivo': 'Medir friccion de servicio percibida por el cliente.',
        'Formula': 'Pedidos con reclamo / total de pedidos',
        'Fuente': 'logiandes.csv: reclamos_registrados, motivo_reclamo',
        'Frecuencia': 'Semanal',
        'Meta': '<= 3%',
        'Umbral': '> 5% requiere accion correctiva',
        'Interpretacion': 'Ayuda a priorizar causas como demora, direccion, producto o atencion.',
    },
    {
        'Nombre': 'Distancia promedio por pedido',
        'Objetivo': 'Entender complejidad fisica de la red logistica.',
        'Formula': 'Promedio de distancia_km',
        'Fuente': 'logiandes.csv: distancia_km',
        'Frecuencia': 'Semanal / mensual',
        'Meta': 'Monitoreo por provincia y canal',
        'Umbral': 'Aumentos con mayor tiempo de entrega deben investigarse',
        'Interpretacion': 'Permite separar demoras por cobertura geografica de demoras por gestion interna.',
    },
    {
        'Nombre': 'Ventas en riesgo logistico',
        'Objetivo': 'Cuantificar valor comercial expuesto a demoras o reclamos.',
        'Formula': 'Suma de ventas_asociadas_usd en pedidos fuera de SLA o con reclamo',
        'Fuente': 'logiandes.csv: ventas_asociadas_usd, reclamos_registrados, tiempo_individual_entrega_min',
        'Frecuencia': 'Semanal / mensual',
        'Meta': 'Reducir tendencia',
        'Umbral': 'Crecimiento sostenido frente al periodo anterior',
        'Interpretacion': 'Conecta decisiones logisticas con impacto comercial y financiero.',
    },
])

ficha_kpis

## 4. Vista ejecutiva

La vista ejecutiva resume volumen, ventas asociadas, cumplimiento de SLA, cola de demoras y reclamos. Estos indicadores permiten detectar rapidamente si la operacion requiere accion.

In [ ]:
resumen_ejecutivo = pd.DataFrame({
    'Indicador': ['Pedidos', 'Ventas asociadas USD', 'Entrega promedio min', 'P90 entrega min', 'Reclamos', 'Fuera de SLA'],
    'Valor': [
        len(datos),
        datos['ventas_asociadas_usd'].sum(),
        datos['tiempo_individual_entrega_min'].mean(),
        datos['tiempo_individual_entrega_min'].quantile(0.90),
        datos['reclamos_registrados'].sum(),
        datos['tiempo_individual_entrega_min'].gt(SLA_MIN).mean(),
    ],
})
resumen_ejecutivo

## 5. Analisis territorial

El analisis territorial compara provincias y regiones para identificar zonas con mayor demora, mayor tasa de reclamo o mayor volumen de pedidos fuera de SLA.

In [ ]:
territorial = (
    datos.groupby(['region', 'provincia'], dropna=False)
    .agg(
        pedidos=('pedido_id', 'count'),
        ventas_usd=('ventas_asociadas_usd', 'sum'),
        entrega_prom_min=('tiempo_individual_entrega_min', 'mean'),
        entrega_p90_min=('tiempo_individual_entrega_min', lambda s: s.quantile(0.90)),
        reclamos=('reclamos_registrados', 'sum'),
        distancia_prom_km=('distancia_km', 'mean'),
        fuera_sla=('tiempo_individual_entrega_min', lambda s: s.gt(SLA_MIN).mean()),
    )
    .reset_index()
)
territorial['tasa_reclamos'] = territorial['reclamos'] / territorial['pedidos']
territorial.sort_values(['fuera_sla', 'entrega_p90_min'], ascending=False)

## 6. Detalle operativo

El detalle operativo prioriza pedidos y segmentos que afectan la logistica del negocio: demoras extremas, reclamos, ventas asociadas y combinaciones provincia-canal-cliente.

In [ ]:
detalle = datos.copy()
detalle['fuera_sla'] = detalle['tiempo_individual_entrega_min'].gt(SLA_MIN)
detalle['score_riesgo'] = (
    detalle['fuera_sla'].astype(int) * 40
    + detalle['tuvo_reclamo'].astype(int) * 35
    + detalle['tiempo_individual_entrega_min'].rank(pct=True) * 15
    + detalle['ventas_asociadas_usd'].rank(pct=True) * 10
)

detalle.sort_values('score_riesgo', ascending=False)[[
    'pedido_id', 'fecha_pedido', 'provincia', 'region', 'canal_entrega', 'tipo_cliente',
    'tiempo_individual_entrega_min', 'distancia_km', 'reclamos_registrados',
    'motivo_reclamo', 'ventas_asociadas_usd', 'score_riesgo'
]].head(20)

In [ ]:
reclamos = (
    datos[datos['tuvo_reclamo']]
    .groupby(['motivo_reclamo', 'provincia'], dropna=False)
    .agg(
        reclamos=('pedido_id', 'count'),
        entrega_prom_min=('tiempo_individual_entrega_min', 'mean'),
        ventas_usd=('ventas_asociadas_usd', 'sum'),
    )
    .reset_index()
    .sort_values(['reclamos', 'ventas_usd'], ascending=False)
)
reclamos.head(15)

## 7. Prototipo Streamlit

El prototipo de dashboard esta desarrollado en `presentacion_logiandes.py` e incluye:

- Vista ejecutiva.
- Analisis territorial.
- Detalle operativo.
- Reclamos.
- Simulador de decision.
- KPIs y Riesgos.
- Datos filtrados.

Para ejecutarlo:

```powershell
streamlit run presentacion_logiandes.py
```

## 8. Reflexion final: riesgos y limites

- **Sobreinteractividad:** demasiados filtros pueden dispersar la lectura. Por eso el dashboard mantiene secciones de negocio y KPIs base visibles.
- **Sesgos visuales:** los rankings deben mostrar volumen y no solo porcentajes, porque una provincia con pocos pedidos puede parecer mas critica de lo que es.
- **Privacidad:** la base trabaja con identificadores de pedido y no requiere exponer datos personales.
- **Gobernanza:** cada KPI necesita formula, fuente, frecuencia, meta y umbral para evitar interpretaciones distintas entre areas.
- **Limites interpretativos:** los datos muestran asociaciones, no causalidad. Una demora puede depender de ruta, flota, capacidad, clima, promesa comercial u otros factores no incluidos en el CSV.

La solucion BI propuesta convierte visualizaciones aisladas en componentes conectados: KPIs ejecutivos, lectura territorial, detalle operativo y documentacion de riesgos para apoyar decisiones logisticas responsables.